# Run Gnani Evon v3.3 on a free Kaggle GPU for MachineMitra

Evon v3.3 is open weights and is **not** on Gnani's API platform yet, so we host it ourselves.

**Before you start (Kaggle settings panel on the right):**
1. *Accelerator* → **GPU T4 x2** (needs a phone-verified Kaggle account).
2. *Internet* → **On**.
3. *Persistence* → Files only is fine.

This uses a community 4-bit GGUF quantization of Evon (`luminenio/gnani-evon-v3.3-30B-A3B-GGUF`, about 19 GB at Q4_K_M),
served with llama.cpp's OpenAI-compatible `llama-server`, and exposed through a free Cloudflare quick tunnel.
The first run takes roughly 20–30 minutes (build + download). Keep this tab open while you demo.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
!df -h /tmp /kaggle/working | tail -2

## 1. Build llama.cpp with CUDA (≈10 min)

In [ ]:
%%bash
set -e
nvidia-smi -L                                   # must list 2 x Tesla T4
nvcc --version | tail -1                        # CUDA compiler must exist

# CMake needs libcuda.so. Its folder differs between CUDA versions, so find it.
CUDA_LIB=$(find /usr/local/cuda/ /usr/local/nvidia /usr/lib /usr/lib64 -name "libcuda.so*" 2>/dev/null | grep -v "\.so\..*\..*\..*" | head -1)
[ -z "$CUDA_LIB" ] && CUDA_LIB=$(find / -name "libcuda.so*" -not -path "/proc/*" 2>/dev/null | head -1)
echo "Using driver library: $CUDA_LIB"
[ -z "$CUDA_LIB" ] && { echo "libcuda.so not found"; exit 1; }
mkdir -p /tmp/cudalib && ln -sf "$CUDA_LIB" /tmp/cudalib/libcuda.so

cd /tmp
[ -d llama.cpp ] || git clone --depth 1 https://github.com/ggml-org/llama.cpp
cd llama.cpp
rm -rf build
cmake -B build -DGGML_CUDA=ON -DCMAKE_CUDA_ARCHITECTURES=75 -DLLAMA_CURL=OFF \
  -DCUDAToolkit_ROOT=/usr/local/cuda \
  -DCUDA_cuda_driver_LIBRARY=/tmp/cudalib/libcuda.so \
  -DCMAKE_LIBRARY_PATH=/tmp/cudalib 2>&1 | tail -5
cmake --build build --config Release -j 4 --target llama-server 2>&1 | tail -3
ls -la build/bin/llama-server


## 2. Download the Evon GGUF
We list the repo's files instead of guessing names, then pick the Q4_K_M file(s). Downloads go to `/tmp` because `/kaggle/working` is limited to 20 GB.

In [ ]:
!pip -q install -U huggingface_hub
from huggingface_hub import list_repo_files, hf_hub_download

REPO = "luminenio/gnani-evon-v3.3-30B-A3B-GGUF"
files = [f for f in list_repo_files(REPO) if f.endswith(".gguf")]
print("\n".join(files))

want = [f for f in files if "Q4_K_M" in f] or [f for f in files if "Q4" in f]
assert want, "No 4-bit file found — pick one from the list above and set want = [...] manually"
paths = [hf_hub_download(REPO, f, local_dir="/tmp/evon") for f in sorted(want)]
MODEL_PATH = paths[0]   # for split files, llama.cpp loads the rest automatically from part 1
print("Using:", MODEL_PATH)

## 3. Start the Evon server
Both T4s are used (`--split-mode layer`). An API key protects your server because the tunnel URL is public.

In [ ]:
import secrets, subprocess, time, requests

EVON_API_KEY = secrets.token_urlsafe(24)
log = open("/tmp/evon_server.log", "w")
server = subprocess.Popen([
    "/tmp/llama.cpp/build/bin/llama-server", "-m", MODEL_PATH,
    "--alias", "evon", "--host", "0.0.0.0", "--port", "8080",
    "-ngl", "99", "--split-mode", "layer", "--ctx-size", "8192",
    "--parallel", "2", "--jinja", "--api-key", EVON_API_KEY,
], stdout=log, stderr=subprocess.STDOUT)

for _ in range(180):
    try:
        if requests.get("http://localhost:8080/health", timeout=2).status_code == 200:
            print("Evon server is up"); break
    except requests.RequestException:
        pass
    time.sleep(5)
else:
    print(open("/tmp/evon_server.log").read()[-3000:])
    raise RuntimeError("Server did not start — see log above")

## 4. Quick test (Hinglish in, JSON out)

In [ ]:
r = requests.post("http://localhost:8080/v1/chat/completions",
    headers={"Authorization": f"Bearer {EVON_API_KEY}"},
    json={"model": "evon", "temperature": 0.3, "max_tokens": 300,
          "chat_template_kwargs": {"enable_thinking": False},
          "messages": [
            {"role": "system", "content": "Return only JSON with keys machine_type, symptoms, urgency."},
            {"role": "user", "content": "CNC machine ka spindle bahut garam ho raha hai, production ruka hua hai"}]},
    timeout=120)
print(r.status_code)
print(r.json()["choices"][0]["message"]["content"])

## 5. Open a public tunnel
Cloudflare quick tunnels are free and need no account. The URL changes every time you restart this cell.

In [ ]:
import re, subprocess, time
!wget -q -nc https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /tmp/cloudflared && chmod +x /tmp/cloudflared
tlog = open("/tmp/tunnel.log", "w")
tunnel = subprocess.Popen(["/tmp/cloudflared", "tunnel", "--no-autoupdate", "--url", "http://localhost:8080"],
                          stdout=tlog, stderr=subprocess.STDOUT)
url = None
for _ in range(60):
    m = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", open("/tmp/tunnel.log").read())
    if m: url = m.group(0); break
    time.sleep(2)
print("Paste these into your .env (local) or Streamlit secrets (deployed):\n")
print(f'EVON_BASE_URL = "{url}"')
print(f'EVON_API_KEY = "{EVON_API_KEY}"')
print('EVON_MODEL = "evon"')

## 6. Keep alive
Run this **only after step 5**, while you record the demo. Kaggle stops idle sessions, and a session lasts up to 12 hours.
**Run the cells one by one** (Shift+Enter), not with Run All.

In [ ]:
# Run this ONLY after step 5 has printed your tunnel link.
import time, requests

def server_ok():
    try:
        return requests.get("http://localhost:8080/health", timeout=5).status_code == 200
    except Exception:
        return False

if not server_ok():
    print("The Evon server is not running yet. Run steps 1-5 first, then run this cell.")
else:
    print("Keeping the session alive. Click the stop button to end.")
    while True:
        print(time.strftime("%H:%M:%S"), "server ok:", server_ok(), flush=True)
        time.sleep(300)
